# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [2]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            if r.status_code in {400, 404}:
                r.raise_for_status()

            if r.status_code in RETRY_STATUSES:
                time.sleep(2 ** attempt)
                continue

            r.raise_for_status()

        except requests.exceptions.RequestException as e:

            if attempt == tries - 1:
                raise e
            time.sleep(2 ** attempt)


    raise Exception("All attempts to fetch weather data failed.")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [3]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False
target_dates = ['2026-09-05', '2026-10-03']

try:
    raw_wx = get_weather('2026-09-05', '2026-10-03')
    raw_wx = raw_wx[raw_wx['time'].isin(target_dates)].copy()
    source_val = 'open-meteo'
except Exception as e:
    print(f"API fetch failed: {e}. Falling back to sample data.")
    used_fallback = True

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [4]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = raw_wx.copy()
    wx['source'] = source_val

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [5]:
# TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})


wx['date'] = pd.to_datetime(wx['date'])
wx = wx.reset_index(drop=True)

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [6]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [7]:
# TODO
wx_sorted = wx.sort_values('precip_mm', ascending=False)

wetter_date = wx_sorted.iloc[0]['date'].strftime('%Y-%m-%d')
difference = wx_sorted.iloc[0]['precip_mm'] - wx_sorted.iloc[1]['precip_mm']

print(f"The weather on {wetter_date} was wetter by {difference:.1f} mm of precipitation.")

The weather on 2026-10-03 was wetter by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [8]:
# TODO
from datetime import datetime

print(f"Endpoint: {BASE}")
print("Coordinates: 38.03, -78.51")
print("Date Range: 2026-09-05 to 2026-10-03")
print(f"Pulled at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Data Source: {'Fallback Sample' if used_fallback else 'Real (Open-Meteo)'}")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date Range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-11 23:53:16
Data Source: Real (Open-Meteo)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

Say get_weather quietly returns an empty DataFrame instead of failing. Nothing breaks, so the pipeline keeps running. Then you merge it with your 200 days of sales data: an inner join wipes out every row, and a left join leaves you with a column of NaNs. You get a blank chart, or a flat “rain has no effect on revenue” result, and the logs look perfectly clean. Nobody finds out the connection dropped during ingestion.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [9]:
# stretch
target_dates = ['2026-09-05', '2026-10-03']
cville_raw = get_weather('2026-09-05', '2026-10-03', lat=38.03, lon=-78.51)
cville_raw['city'] = 'Charlottesville'
orlando_raw = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
orlando_raw['city'] = 'Orlando'
stretch_wx = pd.concat([cville_raw, orlando_raw], ignore_index=True)
stretch_wx = stretch_wx[stretch_wx['time'].isin(target_dates)].copy()
stretch_wx = stretch_wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
stretch_wx['date'] = pd.to_datetime(stretch_wx['date'])

stretch_wx = stretch_wx.reset_index(drop=True)

stretch_wx

,date,temp_max,precip_mm,city
0,2026-09-05,29.4,2.5,Charlottesville
1,2026-10-03,22.0,14.7,Charlottesville
2,2026-09-05,32.1,3.9,Orlando
3,2026-10-03,30.4,14.5,Orlando
